In [4]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd

if os.path.isdir("../data/"):
    os.chdir("../")

from scripts.project_config import get_politician2party

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# description of channels

In [5]:
nm_yt_channels = pd.read_json('data/youtube/channels/news_channels.json')
pp_yt_channels = pd.read_json('data/youtube/channels/pp_channels.json')
nm_tt_channels = pd.read_json('data/tiktok/channels/news_channels.json')
pp_tt_channels = pd.read_json('data/tiktok/channels/pp_channels.json')

In [6]:
pp_tt_channels.groupby("party").count()

,username,bio_description,display_name,follower_count,following_count,is_verified,likes_count,video_count,collected_at,name_standard,orientation
party,,,,,,,,,,,
DD,1,1,1,1,1,1,1,1,1,1,1
DG,1,1,1,1,1,1,1,1,1,1,1
DLF,1,1,1,1,1,1,1,1,1,1,1
EELV,5,5,5,5,5,5,5,5,5,5,5
HOR,2,2,2,2,2,2,2,2,2,2,2
LFI,21,21,21,21,21,21,21,21,21,21,21
LO,2,2,2,2,2,2,2,2,2,2,2
LP,1,1,1,1,1,1,1,1,1,1,1
LR,6,6,6,6,6,6,6,6,6,6,6


In [7]:
print(f"{len(pp_tt_channels[pp_tt_channels.is_verified])}/{len(pp_tt_channels)} verified pp channels ({
100*len(pp_tt_channels[pp_tt_channels.is_verified])/len(pp_tt_channels):.2f}%)")
print(f"{len(nm_tt_channels[nm_tt_channels.is_verified])}/{len(nm_tt_channels)} verified nm channels ({
100*len(nm_tt_channels[nm_tt_channels.is_verified])/len(nm_tt_channels):.2f}%)")

34/85 verified pp channels (40.00%)
25/32 verified nm channels (78.12%)


# number of videos per party per year per platform

In [8]:
yt_nm_videos_2022 = pd.read_json("data/youtube/videos/news_videos_2022.jsonl", lines=True)
yt_nm_videos_2024 = pd.read_json("data/youtube/videos/news_videos_2024.jsonl", lines=True)
yt_pp_videos_2022 = pd.read_json("data/youtube/videos/pp_videos_2022.jsonl", lines=True)
yt_pp_videos_2024 = pd.read_json("data/youtube/videos/pp_videos_2024.jsonl", lines=True)

In [9]:
tt_nm_videos_2022 = pd.read_json("data/tiktok/videos/news_videos_2022.jsonl", lines=True)
tt_nm_videos_2024 = pd.read_json("data/tiktok/videos/news_videos_2024.jsonl", lines=True)
tt_pp_videos_2022 = pd.read_json("data/tiktok/videos/pp_videos_2022.jsonl", lines=True)
tt_pp_videos_2024 = pd.read_json("data/tiktok/videos/pp_videos_2024.jsonl", lines=True)

In [10]:
print(f"YOUTUBE: {len(yt_nm_videos_2024)} NM videos, {len(yt_pp_videos_2024)} PP videos, {len(yt_nm_videos_2024) + len(yt_pp_videos_2024)} all videos")
print(f"TIKTOK {len(tt_nm_videos_2024)} NM videos, {len(tt_pp_videos_2024)} PP videos, {len(tt_nm_videos_2024) + len(tt_pp_videos_2024)} all videos")

YOUTUBE: 39112 NM videos, 2520 PP videos, 41632 all videos
TIKTOK 6949 NM videos, 3946 PP videos, 10895 all videos


In [11]:
yt_pp_videos_2022['party'] = yt_pp_videos_2022['name_standard'].apply(lambda x: get_politician2party()[x])
yt_pp_videos_2024['party'] = yt_pp_videos_2024['name_standard'].apply(lambda x: get_politician2party()[x])
tt_pp_videos_2022['party'] = tt_pp_videos_2022['name_standard'].apply(lambda x: get_politician2party()[x])
tt_pp_videos_2024['party'] = tt_pp_videos_2024['name_standard'].apply(lambda x: get_politician2party()[x])

In [12]:
yt_2022_videos_per_party = yt_pp_videos_2022.groupby('party')['videoId'].count().to_frame().rename(columns={'videoId':'count_yt_2022'})
yt_2024_videos_per_party = yt_pp_videos_2024.groupby('party')['videoId'].count().to_frame().rename(columns={'videoId':'count_yt_2024'})
tt_2022_videos_per_party = tt_pp_videos_2022.groupby('party')['id'].count().to_frame().rename(columns={'id':'count_tt_2022'})
tt_2024_videos_per_party = tt_pp_videos_2024.groupby('party')['id'].count().to_frame().rename(columns={'id':'count_tt_2024'})

In [13]:
yt_2022_videos_per_party.merge(
    yt_2024_videos_per_party, on='party', how='outer').merge(
    tt_2022_videos_per_party, on='party', how='outer').merge(
    tt_2024_videos_per_party, on='party', how='outer')

,count_yt_2022,count_yt_2024,count_tt_2022,count_tt_2024
party,,,,
DG,NaN,NaN,NaN,11.0
DLF,131.0,105.0,30.0,106.0
EELV,43.0,98.0,70.0,177.0
LFI,307.0,601.0,459.0,1240.0
LO,194.0,39.0,17.0,199.0
LP,151.0,161.0,9.0,68.0
LR,NaN,12.0,33.0,139.0
MoDem,38.0,84.0,NaN,NaN
NPA,71.0,77.0,NaN,44.0


# description of transcripts 

In [14]:
n_disabled = []
n_working = []
for filename, yt_transcript in zip(["news_2022", "news_2024", "pp_2022", "pp_2024"],
                                   [yt_nm_videos_2022, yt_nm_videos_2024, 
                                    yt_pp_videos_2022, yt_pp_videos_2024]):
    n_disabled.append(yt_transcript.groupby("error_transcript").count().loc['[transcripts_disabled]','videoId'])
    n_working.append(sum(yt_transcript['error_transcript'].isna())+yt_transcript.groupby("error_transcript").count().loc[
        '[transcripts_disabled]','videoId'])
    print(f"{filename}: {n_disabled[-1]}/{n_working[-1]} ({n_disabled[-1]/n_working[-1]})")
    
sum(n_disabled)/sum(n_working)

news_2022: 4986/36519 (0.13653166844656206)


news_2024: 7635/38841 (0.19657063412373524)
pp_2022: 164/1315 (0.1247148288973384)
pp_2024: 320/2514 (0.12728719172633254)


np.float64(0.16549015646112464)

In [15]:
n_nan = []
n_all = []
for filename, tt_transcript in zip(["news_2022", "news_2024", "pp_2022", "pp_2024"],
                                   [tt_nm_videos_2022, tt_nm_videos_2024, tt_pp_videos_2022, tt_pp_videos_2024]):
    n_nan.append(sum(tt_transcript['voice_to_text'].isna()))
    n_all.append(len(tt_transcript))
    print(f"{filename}: {n_nan[-1]}/{n_all[-1]} ({n_nan[-1]/n_all[-1]})")
sum(n_nan)/sum(n_all)

news_2022: 0/1758 (0.0)
news_2024: 0/6949 (0.0)
pp_2022: 0/809 (0.0)
pp_2024: 1/3946 (0.00025342118601115053)


7.42831674342594e-05

In [17]:
tt_pp_videos_2024[tt_pp_videos_2024['voice_to_text'].isna()]

,comment_count,id,username,video_duration,region_code,share_count,video_description,create_time,favorites_count,hashtag_names,like_count,music_id,view_count,voice_to_text,effect_info_list,name_standard,party
2900,14,7383330158469500192,garrido.raquel,20,FR,0,Suis en campagne mais je prends le temps d'all...,2024-06-22 14:12:04,1,[],35,NaN,3481,NaN,None,Raquel Garrido,LFI


# statistics on transcripts presence

In [18]:
yt_videos_df = pd.concat([yt_nm_videos_2022, yt_nm_videos_2024, yt_pp_videos_2022, yt_pp_videos_2024])

In [41]:
yt_nm_videos_2024

,videoId,publishedAt,channelId,title,description,channelTitle,playlistId,duration,snippet.tags,snippet.categoryId,...,Jordan Bardella,Rima Hassan,Marine Le Pen,Manon Aubry,Marion Maréchal,Emmanuel Macron,Gabriel Attal,transcript,error_transcript,name_standard
0,w4jiApM6epY,2024-07-11T09:34:48Z,UCQd5LYcykaFeZtjDaP3bNSA,Déserts médicaux : la proposition choc du Nouv...,C'est une problématique à laquelle beaucoup de...,Le Point,UUQd5LYcykaFeZtjDaP3bNSA,118,Déserts médicaux#Guillaume Garot#Médecins#NFP,26,...,False,False,False,False,False,False,False,,[transcripts_disabled],Le Point
1,-8efocyKYKU,2024-06-23T10:00:34Z,UCQd5LYcykaFeZtjDaP3bNSA,Législatives : que contient le programme du No...,"La gauche apparaissait divisée, voire déchirée...",Le Point,UUQd5LYcykaFeZtjDaP3bNSA,93,NFP#dissolution#légilsatives#programme,26,...,False,False,False,False,False,False,False,,[transcripts_disabled],Le Point
2,1BvVzAQionI,2024-06-21T12:33:41Z,UCQd5LYcykaFeZtjDaP3bNSA,Jean-Luc Mélenchon règle ses comptes avec Fran...,Dans un entretien accordé au Figaro ce vendred...,Le Point,UUQd5LYcykaFeZtjDaP3bNSA,92,François RUffin#Jean-Luc Mélenchon#LFI#NFP,26,...,False,False,False,False,False,False,False,oh [Music] oh [Applause] [Music] [Music],NaN,Le Point
3,-cQ--R0l6OQ,2024-06-15T09:18:54Z,UCQd5LYcykaFeZtjDaP3bNSA,"L’amour secret d’Eva Braun, la maîtresse de Hi...",Qui était vraiment Eva Braun ? Un nouveau témo...,Le Point,UUQd5LYcykaFeZtjDaP3bNSA,103,Allemagne#Berghof#Christa Schroeder#Eva Braun#...,26,...,False,False,False,False,False,False,False,,[transcripts_disabled],Le Point
4,2sfyrCZ-4Jc,2024-06-05T08:54:29Z,UCQd5LYcykaFeZtjDaP3bNSA,Pluie et ouragans : le pire est à venir,L'année 2023 ayant été la plus chaude jamais e...,Le Point,UUQd5LYcykaFeZtjDaP3bNSA,108,Cyclone#Réchauffement climatique#Tempête,26,...,False,False,False,False,False,False,False,[సంగీతం] ఆ [సంగీతం] اللہ [సంగీతం] ఆ [సంగీతం] [...,NaN,Le Point
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39107,pxwXpbwBA8s,2024-07-02T09:13:38Z,UCCCPCZNChQdGa9EkATeye4g,Euro-2024 : Les Bleus filent en quart de final...,La France a dominé d'une courte tête la Belgiq...,FRANCE 24,UUCCPCZNChQdGa9EkATeye4g,95,LesBleus#euro2024#football#sport,25,...,False,False,False,False,False,False,False,le R de football le Portugal a battu la Slovén...,NaN,FRANCE 24
39108,PlmWSM0QgDI,2024-03-06T19:53:21Z,UCejTWYquMXD_MFHRZiqo_Aw,"Faada Freddy, le corps pour seul instrument - ...",C à vous 👉 https://bit.ly/CaVousReplayC à vous...,C à vous,UUejTWYquMXD_MFHRZiqo_Aw,468,Bertrand Chameroy#Faada Freddy#Mohamed Bouhafs...,24,...,False,False,False,False,False,False,False,NaN,[transcripts_disabled],C à vous
39109,xhWcfJ0YZG4,2024-05-09T08:22:42Z,UCIMGfEAERXjmWwQeg15BFsg,Séparation avec Laure Manaudou : «C'est une le...,Invité dans Culture médias à l'occasion de la ...,Europe 1,UUIMGfEAERXjmWwQeg15BFsg,158,actu#actualité#actus#culture#europ1#europe 1#e...,25,...,False,False,False,False,False,False,False,avec est votre invité ce matin Thomas vous rec...,NaN,Europe 1
39110,4x4MiX1Vm-E,2024-07-03T15:28:19Z,UCESTwDXpoMgiYBHipMdKTkQ,"L'érotisme, un allié important du désir - le L...",Retrouvez Brigitte Lahaie du lundi au vendredi...,Sud Radio,UUESTwDXpoMgiYBHipMdKTkQ,389,NaN,26,...,False,False,False,False,False,False,False,cam4.fr le plus grand site de webcam live rése...,NaN,Sud Radio


In [19]:
yt_transcripts_2024_grp_by_channel = yt_nm_videos_2024.groupby('channelTitle').count()
yt_transcripts_2024_grp_by_channel['% error'] = 100* yt_transcripts_2024_grp_by_channel['error_transcript'] / \
                                                yt_transcripts_2024_grp_by_channel['videoId']
yt_transcripts_2024_grp_by_channel['% error']

channelTitle
AFP                8.663533
BFMTV             10.042918
Blast              0.598802
C dans l'air      97.444634
C à vous          57.400000
CNEWS              1.055409
Euronews           5.788776
Europe 1           0.035759
FRANCE 24          0.353201
France Inter      97.622513
L'Express          0.561798
L'Humanité         0.480769
LCI                0.105042
LCP                0.000000
Le Figaro         15.719947
Le Monde          18.235294
Le Média           0.349650
Le Nouvel Obs    100.000000
Le Parisien        7.013738
Le Point          60.619872
LeHuffPost        25.290023
Les Echos         31.304348
Libération         0.000000
Marianne           0.000000
Mediapart          4.385965
Public Sénat       0.497018
RFI              100.000000
RMC                0.205339
RTL              100.000000
Sud Radio          1.001753
TF1 INFO           0.946074
TV5MONDE Info      0.574506
VA Plus            0.653595
franceinfo         0.352113
Name: % error, dtype: float64

# Statistically significant channels for per channel stats

We retained only channels whose per-channel proportion could be estimated with a 95% margin of error no wider than ±20 percentage points, which requires a minimum of 24 videos. 

In [20]:
video_count_yt_nm_2024 = yt_nm_videos_2024.groupby('name_standard').count()[['videoId']].rename(columns={'videoId':'video_count'})
video_count_yt_pp_2024 = yt_pp_videos_2024.groupby('name_standard').count()[['videoId']].rename(columns={'videoId':'video_count'})
video_count_tt_nm_2024 = tt_nm_videos_2024.groupby('name_standard').count()[['id']].rename(columns={'id':'video_count'})
video_count_tt_pp_2024 = tt_pp_videos_2024.groupby('name_standard').count()[['id']].rename(columns={'id':'video_count'})

In [39]:
for df_spec, df in zip(["YouTube News Media 2024", "YouTube Politicians and Parties 2024",
               "TikTok News Media 2024", "TikTok Politicians and Parties 2024"],
              [video_count_yt_nm_2024, video_count_yt_pp_2024, video_count_tt_nm_2024, video_count_tt_pp_2024]):
    print(df_spec)
    channels_w_too_few_vids = df[df['video_count'] < 24]
    print(f"-> Channels that should not be used, because too few videos:{list(channels_w_too_few_vids.index)}\n")
    if len(df[df['video_count'] < 24]) > 0:
        print(f"details: {dict(zip(df[df['video_count'] < 24].index,df[df['video_count'] < 24]['video_count']))}\n")
    print(40*"-" + "\n")

YouTube News Media 2024
-> Channels that should not be used, because too few videos:[]

----------------------------------------

YouTube Politicians and Parties 2024
-> Channels that should not be used, because too few videos:['Gabriel Attal', 'LP', 'LR', 'RE', 'Rima Hassan', 'UDI']

details: {'Gabriel Attal': 4, 'LP': 6, 'LR': 12, 'RE': 2, 'Rima Hassan': 20, 'UDI': 23}

----------------------------------------

TikTok News Media 2024
-> Channels that should not be used, because too few videos:['Public Sénat']

details: {'Public Sénat': 12}

----------------------------------------

TikTok Politicians and Parties 2024
-> Channels that should not be used, because too few videos:['Alexis Corbière', 'Aurore Bergé', 'Aymeric Caron', 'Boris Vallaud', 'David Rachline', 'EELV', 'Eric Coquerel', 'François Hollande', 'Ian Brossat Sénateur', 'Julien Sanchez', 'Karl Olive', 'LR', 'Laure Lavalette', 'Laurent Jacobelli', 'Louis_Aliot', 'Matthieu Valet', 'Olivier Besancenot', 'Olivier Faure', 'Oliv